In [4]:
import torch

state_dict = torch.load("../../../logs/baseline/final_model.pt")
for k, v in state_dict.items():
    print(f"{k:30s} {v.shape}")

skip_weights                   torch.Size([4, 256])
tok_emb.weight                 torch.Size([1024, 256])
blocks.0.attn_scale            torch.Size([256])
blocks.0.mlp_scale             torch.Size([256])
blocks.0.resid_mix             torch.Size([2, 256])
blocks.0.attn.q_gain           torch.Size([8])
blocks.0.attn.c_q.weight       torch.Size([256, 256])
blocks.0.attn.c_k.weight       torch.Size([128, 256])
blocks.0.attn.c_v.weight       torch.Size([128, 256])
blocks.0.attn.proj.weight      torch.Size([256, 256])
blocks.0.mlp.fc.weight         torch.Size([512, 256])
blocks.0.mlp.proj.weight       torch.Size([256, 512])
blocks.1.attn_scale            torch.Size([256])
blocks.1.mlp_scale             torch.Size([256])
blocks.1.resid_mix             torch.Size([2, 256])
blocks.1.attn.q_gain           torch.Size([8])
blocks.1.attn.c_q.weight       torch.Size([256, 256])
blocks.1.attn.c_k.weight       torch.Size([128, 256])
blocks.1.attn.c_v.weight       torch.Size([128, 256])
blocks.1.attn

In [5]:
for k, t in state_dict.items():
    if t.ndim != 2: continue
    if min(t.shape) < 10: continue

    svals = torch.linalg.svdvals(t.float())
    energy = svals ** 2
    rank = torch.arange(len(svals), device="cuda")[energy.cumsum(dim=0) > 0.9 * energy.sum()][0].item()

    params = t.shape[0] * t.shape[1]
    low_rank_params = (t.shape[0] * rank) + (t.shape[1] * rank)
    print(f"{k:30s} {tuple(t.shape)}     {rank}    {rank / min(t.shape):5f}    {params}    {low_rank_params}      {low_rank_params / params}")

tok_emb.weight                 (1024, 256)     169    0.660156    262144    216320      0.8251953125
blocks.0.attn.c_q.weight       (256, 256)     125    0.488281    65536    64000      0.9765625
blocks.0.attn.c_k.weight       (128, 256)     78    0.609375    32768    29952      0.9140625
blocks.0.attn.c_v.weight       (128, 256)     86    0.671875    32768    33024      1.0078125
blocks.0.attn.proj.weight      (256, 256)     130    0.507812    65536    66560      1.015625
blocks.0.mlp.fc.weight         (512, 256)     172    0.671875    131072    132096      1.0078125
blocks.0.mlp.proj.weight       (256, 512)     178    0.695312    131072    136704      1.04296875
blocks.1.attn.c_q.weight       (256, 256)     128    0.500000    65536    65536      1.0
blocks.1.attn.c_k.weight       (128, 256)     80    0.625000    32768    30720      0.9375
blocks.1.attn.c_v.weight       (128, 256)     83    0.648438    32768    31872      0.97265625
blocks.1.attn.proj.weight      (256, 256)     131   